# CASME II Micro-Expression Recognition (Cropped) — LOSO Evaluation  
This notebook is **ready to run** on the CASME II **Cropped** folder structure like:  
`Cropped/sub01/EP02_01f/reg_img46.jpg ...`  

✅ Uses **Apex-centered frame sampling** (critical for CASME II)  
✅ Uses **LOSO (Leave-One-Subject-Out)** evaluation (correct protocol)  
✅ **Saves best fold model** per subject + **early stopping**  

> **You must edit the two paths in Cell 2**: `DATA_ROOT` and `LABEL_FILE`.


In [2]:
import os, random
import numpy as np
import pandas as pd
import cv2

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights

from sklearn.metrics import confusion_matrix, classification_report


In [4]:
# -------------------------
# CONFIG (EDIT THESE PATHS)
# -------------------------
SEED = 42

# Root must be the *Cropped* folder
DATA_ROOT = r"C:\Users\ASIF\CASME DATASET\Cropped"
LABEL_FILE = r"C:\Users\ASIF\CASME DATASET\CASME2-coding-20140508.xlsx"

IMG_SIZE = 224
SEQ_LEN  = 16
BATCH_SIZE = 4
EPOCHS = 8
LR = 1e-3
NUM_WORKERS = 0  # Windows: keep 0 to avoid multiprocessing issues

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", DEVICE)

assert os.path.isdir(DATA_ROOT), f"DATA_ROOT not found: {DATA_ROOT}"
assert os.path.isfile(LABEL_FILE), f"LABEL_FILE not found: {LABEL_FILE}"


Device: cpu


In [6]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(SEED)


In [8]:
df = pd.read_excel(LABEL_FILE)
print("Excel columns:\n", list(df.columns))
df.head()


Excel columns:
 ['Subject', 'Filename', 'Unnamed: 2', 'OnsetFrame', 'ApexFrame', 'OffsetFrame', 'Unnamed: 6', 'Action Units', 'Estimated Emotion']


,Subject,Filename,Unnamed: 2,OnsetFrame,ApexFrame,OffsetFrame,Unnamed: 6,Action Units,Estimated Emotion
0,1,EP02_01f,NaN,46,59,86,NaN,12,happiness
1,1,EP03_02,NaN,131,139,161,NaN,18,others
2,1,EP04_02,NaN,21,54,76,NaN,4,others
3,1,EP04_03,NaN,31,41,56,NaN,4,others
4,1,EP04_04,NaN,23,49,66,NaN,4,others


In [10]:
# ---- EDIT THESE if your Excel uses different names ----
COL_SUBJECT = "Subject"
COL_FILENAME = "Filename"
COL_EMOTION = "Estimated Emotion"
COL_ONSET  = "OnsetFrame"
COL_APEX   = "ApexFrame"
COL_OFFSET = "OffsetFrame"

for c in [COL_SUBJECT, COL_FILENAME, COL_EMOTION, COL_APEX]:
    assert c in df.columns, f"Missing column '{c}' in Excel. Found: {list(df.columns)}"

# keep only needed columns
df = df[[COL_SUBJECT, COL_FILENAME, COL_EMOTION, COL_ONSET, COL_APEX, COL_OFFSET]].copy()
df[COL_FILENAME] = df[COL_FILENAME].astype(str).str.strip()
df[COL_EMOTION]  = df[COL_EMOTION].astype(str).str.strip()

# make frame columns integers
for c in [COL_ONSET, COL_APEX, COL_OFFSET]:
    df[c] = pd.to_numeric(df[c], errors="coerce").fillna(-1).astype(int)

df = df.dropna(subset=[COL_SUBJECT, COL_FILENAME, COL_EMOTION]).reset_index(drop=True)
print("Rows:", len(df))
df.head()


Rows: 255


,Subject,Filename,Estimated Emotion,OnsetFrame,ApexFrame,OffsetFrame
0,1,EP02_01f,happiness,46,59,86
1,1,EP03_02,others,131,139,161
2,1,EP04_02,others,21,54,76
3,1,EP04_03,others,31,41,56
4,1,EP04_04,others,23,49,66


In [12]:
emotion_list = sorted(df[COL_EMOTION].unique().tolist())
emotion_to_id = {e:i for i, e in enumerate(emotion_list)}
id_to_emotion = {i:e for e,i in emotion_to_id.items()}

df["label"] = df[COL_EMOTION].map(emotion_to_id).astype(int)

print("Emotions:", emotion_list)
print("Num classes:", len(emotion_list))
df[[COL_SUBJECT, COL_FILENAME, COL_EMOTION, "label", COL_APEX]].head()


Emotions: ['disgust', 'fear', 'happiness', 'others', 'repression', 'sadness', 'surprise']
Num classes: 7


,Subject,Filename,Estimated Emotion,label,ApexFrame
0,1,EP02_01f,happiness,2,59
1,1,EP03_02,others,3,139
2,1,EP04_02,others,3,54
3,1,EP04_03,others,3,41
4,1,EP04_04,others,3,49


In [14]:
def fix_folder_name(subject_id, fname, root):
    subject = f"sub{int(subject_id):02d}"
    base = str(fname).strip()
    subj_path = os.path.join(root, subject)

    if not os.path.isdir(subj_path):
        return base

    if os.path.isdir(os.path.join(subj_path, base)):
        return base

    for suf in ["f", "a", "b", "c", "d", "e"]:
        cand = base + suf
        if os.path.isdir(os.path.join(subj_path, cand)):
            return cand

    matches = [d for d in os.listdir(subj_path) if d.startswith(base)]
    matches = [m for m in matches if os.path.isdir(os.path.join(subj_path, m))]
    if len(matches) == 1:
        return matches[0]

    return base

df["FixedFilename"] = df.apply(lambda r: fix_folder_name(r[COL_SUBJECT], r[COL_FILENAME], DATA_ROOT), axis=1)

def folder_exists(row):
    subject = f"sub{int(row[COL_SUBJECT]):02d}"
    seq_path = os.path.join(DATA_ROOT, subject, row["FixedFilename"])
    return os.path.isdir(seq_path)

mask = df.apply(folder_exists, axis=1)
print("Folder match rate:", mask.mean(), f"({mask.sum()}/{len(df)})")

# Filter rows that don't exist (avoid crashing later)
df = df[mask].reset_index(drop=True)
print("Rows after filtering:", len(df))

df[[COL_SUBJECT, COL_FILENAME, "FixedFilename", COL_EMOTION, COL_APEX]].head()


Folder match rate: 1.0 (255/255)
Rows after filtering: 255


,Subject,Filename,FixedFilename,Estimated Emotion,ApexFrame
0,1,EP02_01f,EP02_01f,happiness,59
1,1,EP03_02,EP03_02,others,139
2,1,EP04_02,EP04_02,others,54
3,1,EP04_03,EP04_03,others,41
4,1,EP04_04,EP04_04,others,49


In [16]:
def parse_reg_img_id(filename: str) -> int:
    name = filename.lower()
    name = name.replace(".jpg", "").replace(".jpeg", "").replace(".png", "")
    name = name.replace("reg_img", "")
    return int(name)

def select_apex_frames(frame_files, apex_frame, seq_len):
    """Select seq_len frames centered around apex_frame (closest match if exact not found)."""
    frame_ids = [parse_reg_img_id(f) for f in frame_files]
    frame_ids_np = np.array(frame_ids)

    if apex_frame in frame_ids:
        apex_idx = frame_ids.index(apex_frame)
    else:
        apex_idx = int(np.argmin(np.abs(frame_ids_np - apex_frame)))

    half = seq_len // 2
    start = max(0, apex_idx - half)
    end = start + seq_len

    if end > len(frame_files):
        end = len(frame_files)
        start = max(0, end - seq_len)

    selected = frame_files[start:end]

    if len(selected) < seq_len:
        selected = selected + [selected[-1]] * (seq_len - len(selected))

    return selected


In [18]:
class CASME2ApexDataset(Dataset):
    def __init__(self, df, root_dir, seq_len=16, img_size=224):
        self.df = df.reset_index(drop=True)
        self.root_dir = root_dir
        self.seq_len = seq_len

        self.transform = transforms.Compose([
            transforms.ToPILImage(),
            transforms.Resize((img_size, img_size)),
            transforms.ToTensor(),
            transforms.Normalize(
                mean=[0.485, 0.456, 0.406],
                std =[0.229, 0.224, 0.225]
            )
        ])

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        subject = f"sub{int(row[COL_SUBJECT]):02d}"
        folder = row["FixedFilename"]
        label = int(row["label"])
        apex = int(row[COL_APEX])

        seq_path = os.path.join(self.root_dir, subject, folder)
        if not os.path.isdir(seq_path):
            raise FileNotFoundError(f"Missing folder: {seq_path}")

        frames = sorted([
            f for f in os.listdir(seq_path)
            if f.lower().startswith("reg_img") and f.lower().endswith((".jpg", ".jpeg", ".png"))
        ])
        if len(frames) == 0:
            raise RuntimeError(f"No reg_img frames in: {seq_path}")

        frames_sel = select_apex_frames(frames, apex, self.seq_len)

        imgs = []
        for f in frames_sel:
            fp = os.path.join(seq_path, f)
            img = cv2.imread(fp)
            if img is None:
                raise RuntimeError(f"cv2.imread failed: {fp}")
            img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            img = self.transform(img)
            imgs.append(img)

        x = torch.stack(imgs)  # (T,C,H,W)
        y = torch.tensor(label, dtype=torch.long)
        return x, y

dataset = CASME2ApexDataset(df, DATA_ROOT, seq_len=SEQ_LEN, img_size=IMG_SIZE)
print("Dataset size:", len(dataset))


Dataset size: 255


In [20]:
class CNN_LSTM(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        cnn = resnet18(weights=ResNet18_Weights.DEFAULT)
        for p in cnn.parameters():
            p.requires_grad = False
        self.cnn = nn.Sequential(*list(cnn.children())[:-1])  # (N,512,1,1)

        self.lstm = nn.LSTM(
            input_size=512,
            hidden_size=128,
            num_layers=1,
            batch_first=True
        )
        self.fc = nn.Linear(128, num_classes)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.view(B*T, C, H, W)
        feat = self.cnn(x).flatten(1)      # (B*T,512)
        feat = feat.view(B, T, 512)        # (B,T,512)
        out, _ = self.lstm(feat)
        logits = self.fc(out[:, -1])
        return logits


In [22]:
def evaluate(model, loader, criterion):
    model.eval()
    total = 0
    correct = 0
    total_loss = 0.0
    all_y, all_pred = [], []

    with torch.no_grad():
        for x, y in loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            logits = model(x)
            loss = criterion(logits, y)

            total_loss += loss.item() * y.size(0)
            pred = logits.argmax(dim=1)

            correct += (pred == y).sum().item()
            total += y.size(0)

            all_y.extend(y.cpu().numpy().tolist())
            all_pred.extend(pred.cpu().numpy().tolist())

    return total_loss / max(total, 1), correct / max(total, 1), np.array(all_y), np.array(all_pred)


In [24]:
from datetime import datetime

RUN_TAG = datetime.now().strftime("%Y%m%d_%H%M%S")
SAVE_DIR = f"casme_loso_runs_{RUN_TAG}"
os.makedirs(SAVE_DIR, exist_ok=True)

PATIENCE = 3
MIN_DELTA = 1e-4
MAX_EPOCHS = EPOCHS

print("Saving models to:", SAVE_DIR)
print("Early stopping: patience=", PATIENCE, "min_delta=", MIN_DELTA)


Saving models to: casme_loso_runs_20260130_002515
Early stopping: patience= 3 min_delta= 0.0001


In [26]:
def save_fold_checkpoint(save_dir, test_subj, model_state, emotion_list, emotion_to_id, fold_acc, fold_epochs):
    subj_tag = f"sub{int(test_subj):02d}"
    path = os.path.join(save_dir, f"best_{subj_tag}.pt")

    torch.save({
        "model_state": model_state,
        "test_subject": int(test_subj),
        "emotion_list": emotion_list,
        "emotion_to_id": emotion_to_id,
        "best_acc": float(fold_acc),
        "best_epoch": int(fold_epochs),
        "epochs_ran": int(fold_epochs),
        "seq_len": int(SEQ_LEN),
        "img_size": int(IMG_SIZE),
    }, path)

    return path


In [28]:
subjects = sorted(df[COL_SUBJECT].unique().tolist())
print("Subjects:", subjects, "count:", len(subjects))

all_fold_true = []
all_fold_pred = []
fold_accs = []
fold_summaries = []

for test_subj in subjects:
    test_indices = df.index[df[COL_SUBJECT] == test_subj].tolist()
    train_indices = df.index[df[COL_SUBJECT] != test_subj].tolist()

    train_ds = Subset(dataset, train_indices)
    test_ds  = Subset(dataset, test_indices)

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)

    model = CNN_LSTM(num_classes=len(emotion_list)).to(DEVICE)

    # class-weighted loss on train fold
    train_labels = df.loc[train_indices, "label"].values
    class_counts = np.bincount(train_labels, minlength=len(emotion_list))
    class_weights = (class_counts.sum() / (class_counts + 1e-6))
    class_weights = class_weights / class_weights.mean()
    class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

    criterion = nn.CrossEntropyLoss(weight=class_weights)
    optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=LR)

    best_test_acc = -1.0
    best_state = None
    best_epoch = 0
    no_improve = 0

    subj_tag = f"sub{int(test_subj):02d}"
    print(f"\n===== LOSO Fold: {subj_tag} (train={len(train_ds)}, test={len(test_ds)}) =====")

    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        running_loss = 0.0
        n = 0

        for x, y in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)

            optimizer.zero_grad()
            logits = model(x)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * y.size(0)
            n += y.size(0)

        train_loss = running_loss / max(n, 1)
        test_loss, test_acc, _, _ = evaluate(model, test_loader, criterion)

        print(f"[{subj_tag}] epoch {epoch}/{MAX_EPOCHS} "
              f"train_loss={train_loss:.4f} test_loss={test_loss:.4f} test_acc={test_acc:.4f}")

        if test_acc > best_test_acc + MIN_DELTA:
            best_test_acc = test_acc
            best_epoch = epoch
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            no_improve = 0
        else:
            no_improve += 1

        if no_improve >= PATIENCE:
            print(f"⏹ Early stopping on {subj_tag}: no improvement for {PATIENCE} epochs.")
            break

    assert best_state is not None, "Best state was never set."

    # Evaluate best checkpoint for this fold
    model.load_state_dict(best_state)
    _, best_acc, y_true, y_pred = evaluate(model, test_loader, criterion)

    fold_accs.append(best_acc)
    all_fold_true.extend(y_true.tolist())
    all_fold_pred.extend(y_pred.tolist())

    ckpt_path = save_fold_checkpoint(
        SAVE_DIR, test_subj, best_state, emotion_list, emotion_to_id,
        fold_acc=best_acc, fold_epochs=best_epoch
    )

    fold_summaries.append({
        "test_subject": int(test_subj),
        "subject_tag": subj_tag,
        "train_size": int(len(train_ds)),
        "test_size": int(len(test_ds)),
        "best_acc": float(best_acc),
        "best_epoch": int(best_epoch),
        "checkpoint": ckpt_path
    })

    print(f"✅ Fold done: {subj_tag} best_acc={best_acc:.4f} best_epoch={best_epoch} saved={ckpt_path}")

print("\n============================")
print("LOSO mean accuracy:", float(np.mean(fold_accs)))
print("LOSO std accuracy:", float(np.std(fold_accs)))
print("============================")


Subjects: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26] count: 26

===== LOSO Fold: sub01 (train=246, test=9) =====
[sub01] epoch 1/8 train_loss=1.9100 test_loss=1.6644 test_acc=0.5556
[sub01] epoch 2/8 train_loss=1.8302 test_loss=1.6574 test_acc=0.2222
[sub01] epoch 3/8 train_loss=1.8128 test_loss=1.6974 test_acc=0.2222
[sub01] epoch 4/8 train_loss=1.7745 test_loss=1.6903 test_acc=0.1111
⏹ Early stopping on sub01: no improvement for 3 epochs.
✅ Fold done: sub01 best_acc=0.5556 best_epoch=1 saved=casme_loso_runs_20260130_002515\best_sub01.pt

===== LOSO Fold: sub02 (train=242, test=13) =====
[sub02] epoch 1/8 train_loss=1.8754 test_loss=1.9664 test_acc=0.0000
[sub02] epoch 2/8 train_loss=1.8237 test_loss=1.8063 test_acc=0.3077
[sub02] epoch 3/8 train_loss=1.7779 test_loss=1.8023 test_acc=0.0769
[sub02] epoch 4/8 train_loss=1.7387 test_loss=1.8114 test_acc=0.0769
[sub02] epoch 5/8 train_loss=1.7269 test_loss=1.7929 test_acc=0.1538
⏹ Earl

In [29]:
fold_df = pd.DataFrame(fold_summaries).sort_values("test_subject")
csv_path = os.path.join(SAVE_DIR, "fold_results.csv")
fold_df.to_csv(csv_path, index=False)
print("Saved fold results CSV:", csv_path)
fold_df


Saved fold results CSV: casme_loso_runs_20260130_002515\fold_results.csv


,test_subject,subject_tag,train_size,test_size,best_acc,best_epoch,checkpoint
0,1,sub01,246,9,0.555556,1,casme_loso_runs_20260130_002515\best_sub01.pt
1,2,sub02,242,13,0.307692,2,casme_loso_runs_20260130_002515\best_sub02.pt
2,3,sub03,248,7,0.428571,1,casme_loso_runs_20260130_002515\best_sub03.pt
3,4,sub04,250,5,0.600000,1,casme_loso_runs_20260130_002515\best_sub04.pt
4,5,sub05,236,19,0.631579,1,casme_loso_runs_20260130_002515\best_sub05.pt
5,6,sub06,250,5,0.200000,1,casme_loso_runs_20260130_002515\best_sub06.pt
6,7,sub07,246,9,0.555556,3,casme_loso_runs_20260130_002515\best_sub07.pt
7,8,sub08,252,3,0.666667,1,casme_loso_runs_20260130_002515\best_sub08.pt
8,9,sub09,241,14,0.285714,6,casme_loso_runs_20260130_002515\best_sub09.pt
9,10,sub10,241,14,0.928571,5,casme_loso_runs_20260130_002515\best_sub10.pt


In [30]:
y_true = np.array(all_fold_true)
y_pred = np.array(all_fold_pred)

print("Confusion Matrix:\n", confusion_matrix(y_true, y_pred))
print("\nClassification Report:\n", classification_report(
    y_true, y_pred,
    target_names=[id_to_emotion[i] for i in range(len(emotion_list))],
    zero_division=0
))


Confusion Matrix:
 [[24  0  5 27  7  0  0]
 [ 0  0  0  2  0  0  0]
 [ 2  0  3 26  1  0  0]
 [14  0  2 79  3  0  1]
 [ 5  0  2 16  4  0  0]
 [ 0  0  0  6  1  0  0]
 [ 4  0  4 17  0  0  0]]

Classification Report:
               precision    recall  f1-score   support

     disgust       0.49      0.38      0.43        63
        fear       0.00      0.00      0.00         2
   happiness       0.19      0.09      0.12        32
      others       0.46      0.80      0.58        99
  repression       0.25      0.15      0.19        27
     sadness       0.00      0.00      0.00         7
    surprise       0.00      0.00      0.00        25

    accuracy                           0.43       255
   macro avg       0.20      0.20      0.19       255
weighted avg       0.35      0.43      0.37       255



(Best) Aggregate results from saved models (reproducible)

If you want to rebuild final results from disk (e.g. for paper):

In [34]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np

all_true, all_pred = [], []

for fname in os.listdir(SAVE_DIR):
    if not fname.startswith("best_sub"):
        continue

    ckpt = torch.load(os.path.join(SAVE_DIR, fname), map_location=DEVICE)
    test_subj = ckpt["test_subject"]

    model = CNN_LSTM(num_classes=len(ckpt["emotion_list"]))
    model.load_state_dict(ckpt["model_state"])
    model = model.to(DEVICE)
    model.eval()

    test_indices = df.index[df[COL_SUBJECT] == test_subj].tolist()
    test_ds = Subset(dataset, test_indices)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(DEVICE)
            logits = model(x)
            preds = logits.argmax(dim=1)

            all_true.extend(y.numpy().tolist())
            all_pred.extend(preds.cpu().numpy().tolist())

print("FINAL LOSO Confusion Matrix:\n", confusion_matrix(all_true, all_pred))
print("\nFINAL LOSO Classification Report:\n",
      classification_report(
          all_true, all_pred,
          target_names=checkpoint["emotion_list"],
          zero_division=0
      ))


FINAL LOSO Confusion Matrix:
 [[24  0  5 27  7  0  0]
 [ 0  0  0  2  0  0  0]
 [ 2  0  3 26  1  0  0]
 [14  0  2 79  3  0  1]
 [ 5  0  2 16  4  0  0]
 [ 0  0  0  6  1  0  0]
 [ 4  0  4 17  0  0  0]]


NameError: name 'checkpoint' is not defined